# Adapter Viewer
Inspect a LoRA adapter: list layers, compute per-matrix singular values (`svd(B @ A)`), visualize energy decay across layers.

#### Imports

In [ ]:
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from safetensors import safe_open

#### Config

In [ ]:
ADAPTER_PATH = Path("../../output/08-21_22-58-Ayush-ground-truth/checkpoint-195")

SAFETENSOR = ADAPTER_PATH / "adapter_model.safetensors"
assert SAFETENSOR.is_file(), f"Not found: {SAFETENSOR}"
print(f"Adapter: {ADAPTER_PATH}")

#### Load tensors and group LoRA pairs
Keys look like `base_model.model.model.layers.{L}.{module}.{lora_A|lora_B}.default.weight`.
Non-LoRA full weights (e.g. `embed_tokens`, `lm_head` when trained directly) are kept separately.

In [ ]:
KEY_RE = re.compile(r"layers\.(\d+)\.[a-z_]+\.(\w+_proj)\.lora_([AB])")

tensors = {}
with safe_open(SAFETENSOR, framework="pt") as f:
    for k in f.keys():
        tensors[k] = f.get_tensor(k).to("cpu").float().numpy()

lora_pairs = defaultdict(dict)  # (layer_idx, module) -> {"A": ..., "B": ...}
full_weights = {}                # non-lora trained weights (embed, lm_head, ...)
for k, w in tensors.items():
    m = KEY_RE.search(k)
    if m:
        layer, module, ab = int(m.group(1)), m.group(2), m.group(3)
        lora_pairs[(layer, module)][ab] = w
    else:
        full_weights[k] = w

print(f"LoRA pairs: {len(lora_pairs)}")
print(f"Full weights: {len(full_weights)}")
modules = sorted({m for _, m in lora_pairs})
layers = sorted({l for l, _ in lora_pairs})
print(f"Modules: {modules}")
print(f"Layers: {layers[0]}..{layers[-1]}  (n={len(layers)})")
for k, w in full_weights.items():
    print(f"  full: {k}  shape={w.shape}")

#### Compute singular values of `ΔW = B @ A`
LoRA update has effective rank at most `r`. `svd(B @ A)` gives the actual spectrum of the update.

In [ ]:
def delta_svs(A, B):
    dW = B @ A                     # (out, in), rank <= r
    return np.linalg.svd(dW, compute_uv=False)

svs = {}  # (layer, module) -> singular values (desc)
for key, pair in lora_pairs.items():
    if "A" not in pair or "B" not in pair:
        continue
    svs[key] = delta_svs(pair["A"], pair["B"])

sample = next(iter(svs.items()))
print(f"Sample {sample[0]}: rank={len(sample[1])}, top5={sample[1][:5].round(4)}")

#### Cumulative energy per layer
Energy = `Σσ²`. Curve rising fast = update concentrated in top singular directions (low effective rank).
Curve rising slowly = update spread over many directions (using more of the allocated rank).

In [ ]:
n_mod = len(modules)
n_col = 3
n_row = (n_mod + n_col - 1) // n_col
fig, axes = plt.subplots(n_row, n_col, figsize=(5.5 * n_col, 3.8 * n_row), squeeze=False)

cmap = plt.get_cmap("viridis")
for idx, mod in enumerate(modules):
    ax = axes[idx // n_col][idx % n_col]
    for L in layers:
        s = svs.get((L, mod))
        if s is None:
            continue
        energy = np.cumsum(s ** 2) / (s ** 2).sum()
        ax.plot(range(1, len(energy) + 1), energy, color=cmap(L / max(layers)), lw=0.8)
    ax.set_title(mod)
    ax.set_xlabel("rank index")
    ax.set_ylabel("cumulative energy")
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.3)

for j in range(n_mod, n_row * n_col):
    axes[j // n_col][j % n_col].axis("off")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=layers[0], vmax=layers[-1]))
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), shrink=0.6, pad=0.02)
cbar.set_label("layer index")
plt.suptitle("Cumulative singular value energy per module (line = layer)", y=1.02)
plt.show()

#### Heatmap: normalized singular values (layer × rank)
Bright column = layer where an update direction is dominant. Bright bottom-right = update spread wide.

In [ ]:
fig, axes = plt.subplots(n_row, n_col, figsize=(5.5 * n_col, 3.8 * n_row), squeeze=False)
for idx, mod in enumerate(modules):
    ax = axes[idx // n_col][idx % n_col]
    rows = []
    for L in layers:
        s = svs.get((L, mod))
        if s is None:
            continue
        rows.append(s / (s[0] + 1e-12))
    mat = np.stack(rows) if rows else np.zeros((1, 1))
    im = ax.imshow(mat, aspect="auto", origin="lower", cmap="magma", vmin=0, vmax=1)
    ax.set_title(f"{mod}   shape={mat.shape}")
    ax.set_xlabel("rank index")
    ax.set_ylabel("layer index")
    fig.colorbar(im, ax=ax, shrink=0.8)

for j in range(n_mod, n_row * n_col):
    axes[j // n_col][j % n_col].axis("off")

plt.suptitle("Singular values normalized by σ₁ per (layer, module)", y=1.02)
plt.tight_layout()
plt.show()

#### Effective rank per layer
Effective rank at 90% / 99% energy — rough proxy for how much of the LoRA budget is actually used.

In [ ]:
THRESHOLDS = [0.90, 0.99]

fig, ax = plt.subplots(figsize=(9, 4.5))
for mod in modules:
    xs, ys90, ys99 = [], [], []
    for L in layers:
        s = svs.get((L, mod))
        if s is None:
            continue
        e = np.cumsum(s ** 2) / (s ** 2).sum()
        xs.append(L)
        ys90.append(int(np.searchsorted(e, 0.90) + 1))
        ys99.append(int(np.searchsorted(e, 0.99) + 1))
    ax.plot(xs, ys90, marker="o", ms=3, lw=1, label=f"{mod} @90%")
    ax.plot(xs, ys99, marker="x", ms=3, lw=1, ls="--", label=f"{mod} @99%")

ax.set_xlabel("layer index")
ax.set_ylabel("effective rank")
ax.set_title("Effective rank per layer (solid=90%, dashed=99%)")
ax.grid(alpha=0.3)
ax.legend(fontsize=7, ncol=2, loc="upper left", bbox_to_anchor=(1.01, 1.0))
plt.tight_layout()
plt.show()